# Luxembourg School Access With `scalable_distances`

This notebook illustrates the promoted API in `Research-Sandbox/scalable_general_distances_per_country`. It uses Luxembourg-style source settings and a tiny in-memory road network so the core API can be inspected quickly without downloading data.

For a real run, use the same API objects with the shared cache at `C:\\local\\Download_Depot` and call the production runner or CLI.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "scalable_distances").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
assert (SRC_ROOT / "scalable_distances").exists(), f"Could not find scalable_distances under {PROJECT_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd

from scalable_distances import describe_backends, describe_country_sources, write_distance_matrix
from scalable_distances.routing.base import NetworkData
from scalable_distances.routing.strategies import NetworkXRouter

## Backend Detection

Backends are detected without importing heavy optional modules such as Pandana. This is useful in deployments where Pandana is unavailable because of NumPy compatibility.

In [ ]:
describe_backends()

## Luxembourg Source Resolution

`describe_country_sources()` resolves the OSM PBF and WorldPop file names, URLs, and shared-cache paths without downloading anything.

In [ ]:
lux_sources = describe_country_sources(
    country_slug="luxembourg",
    iso3="LUX",
    base_dir=Path(r"C:\\local\\Download_Depot\\luxembourg_data"),
    worldpop_dataset="global1",
    worldpop_year=2020,
)
lux_sources

## Shortest Paths Without Pandana

The default portable routing strategy is NetworkX. Pandana is imported only when `PandanaRouter` is selected.

In [ ]:
network = NetworkData(
    nodes=pd.DataFrame([
        {"node_id": 1, "lon": 6.10, "lat": 49.60},
        {"node_id": 2, "lon": 6.12, "lat": 49.61},
        {"node_id": 3, "lon": 6.14, "lat": 49.62},
    ]),
    edges=pd.DataFrame([
        {"u": 1, "v": 2, "length_m": 1200.0},
        {"u": 2, "v": 3, "length_m": 900.0},
        {"u": 2, "v": 1, "length_m": 1200.0},
        {"u": 3, "v": 2, "length_m": 900.0},
    ]),
)

schools = pd.DataFrame([
    {"source_id": "school_centre", "source_type": "amenities", "lon": 6.10, "lat": 49.60},
])
population = pd.DataFrame([
    {"target_id": "pop_north", "target_type": "population", "lon": 6.14, "lat": 49.62, "population": 42.0},
])

router = NetworkXRouter()
router.prepare(network, {})
snapped_schools = router.snap(schools)
snapped_population = router.snap(population)
matrix = router.route_many(snapped_schools, snapped_population)
matrix

## Matrix Outputs

The same table can be written as one combined matrix, split matrices by source/target layer, or both.

In [ ]:
outputs = write_distance_matrix(
    matrix,
    output_dir=PROJECT_ROOT / "diagnostics" / "luxembourg_api_illustration",
    run_tag="luxembourg_api",
    mode="both",
)
{key: str(path) for key, path in outputs.paths.items()}

## Full Production Run

For real data, use `scalable_distances.pipeline.ProductionRunConfig` with `run_production_country()`, or the CLI command:

```powershell
py -m scalable_distances.cli run `
  --country-slug luxembourg `
  --iso3 LUX `
  --base-dir C:\\local\\Download_Depot\\luxembourg_data `
  --output-dir C:\\local\\Download_Depot\\luxembourg_data\\outputs `
  --run-tag luxembourg_networkx `
  --amenity school `
  --router networkx `
  --matrix-output-mode both
```

Switch to `--router pandana` only in an environment where Pandana is installed and compatible.